# 08 — Cross-preprocessing robustness and resource benchmark

Two analyses are kept separate: **A)** resource cost of the frozen primary-dataset method, and **B)** optional QF75/QF85/QF95 preprocessing-history robustness. Variant experiments preserve the primary `source_id → split` mapping to prevent leakage. Cross-source/preprocessing results are secondary and must not be used to retune the frozen primary experiment.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd, numpy as np, yaml, psutil, os
from rdhlab.io import read_gray
from rdhlab.pipeline import load_payload_freeze, prepare_image_context, run_frozen_image_precomputed

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text()); seed=int(config['project']['seed']); bs=int(config['dataset']['block_size'])
manifest=pd.read_csv(config['dataset']['prepared_manifest']); test=manifest[manifest.split=='test'].reset_index(drop=True)
risk=joblib.load('/workspace/results/models/block_risk.joblib'); payloads=list(map(float,load_payload_freeze('/workspace/config/frozen_payloads.json')['levels'])); alpha=float(json.loads(Path('/workspace/config/frozen_allocator.json').read_text())['alpha'])
print('RSS before benchmark: %.1f MiB'%(psutil.Process(os.getpid()).memory_info().rss/1024**2))

In [ ]:
# Resource benchmark on a deterministic 200-image subset at the middle payload.
bpp=payloads[len(payloads)//2]; rows=[]
for i,row in test.head(200).iterrows():
    x=read_gray(row.path); sid=str(row.source_id); orders,br,plans=prepare_image_context(x,sid,risk,alpha,bs,seed)
    for strategy in config['allocator']['strategies']:
        rr=run_frozen_image_precomputed(x,sid,bpp,strategy,orders,br,bs,seed,plans=plans)
        rows.append({k:v for k,v in rr.items() if k!='stego'})
res=pd.DataFrame(rows); res.to_csv('/workspace/results/resource_benchmark.csv',index=False)
display(res.groupby('strategy')[['encode_ms','decode_ms','used_blocks']].agg(['median','mean','std']))
print('RSS after benchmark: %.1f MiB'%(psutil.Process(os.getpid()).memory_info().rss/1024**2))

In [ ]:
# OPTIONAL preprocessing-history experiment. Disabled by default.
RUN_CROSS_SOURCE=bool(config['cross_source']['enabled_by_default'])
print('RUN_CROSS_SOURCE =',RUN_CROSS_SOURCE)
if RUN_CROSS_SOURCE:
    from rdhlab.dataset import find_bossbase_archive, prepare_bossbase
    raw=Path('/workspace/data/raw'); archive=find_bossbase_archive(raw); primary_map=manifest.set_index(manifest.source_id.astype(str))['split'].to_dict()
    variant_manifests=[]
    for variant in config['cross_source']['variants']:
        tag=variant.replace('/','_')
        result=prepare_bossbase(raw,Path('/workspace/data/processed')/tag,archive,10_000,None,20260916,source_subdir=variant,allow_lossy_source=True)
        vm=pd.read_csv(result.manifest_path); vm['source_id']=vm.source_id.astype(str); vm['split']=vm.source_id.map(primary_map)
        assert vm['split'].notna().all(); vm.to_csv(Path('/workspace/data/processed')/f'{tag}_aligned_manifest.csv',index=False)
        variant_manifests.append((variant,vm))
    print('Prepared aligned variants:',[v for v,_ in variant_manifests])

If cross-preprocessing is enabled, run the frozen encoder/weights **without retuning** on each aligned variant and report the performance drop. This is a robustness experiment, not a new training split.